In [1]:
import numpy as np

from scipy.sparse import eye, lil_matrix, diags
from joblib import Parallel, delayed
from resource_estimate_utils import *
from os.path import join
from time import time

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis import LieTrotter, SuzukiTrotter
from qiskit import transpile
from qiskit.circuit.library import PauliEvolutionGate
from pytket import OpType
from pytket.passes import RemoveRedundancies, CommuteThroughMultis, SequencePass, FullPeepholeOptimise, auto_rebase_pass
from pytket.extensions.qiskit import qiskit_to_tk

from qclib.gates.ldmcsu import Ldmcsu

from os.path import join
import sys
sys.path.append(join(".", ".."))
from utils import *

In [5]:
def get_mcrz(n, theta):
    circ = QuantumCircuit(n)
    if n == 2:
        circ.crz(theta, control_qubit=0, target_qubit=1)
    else:
        su2_matrix = np.array([[np.exp(-1.j * theta / 2.), 0.], [0., np.exp(1.j * theta / 2.)]])

        circ.append(Ldmcsu(su2_matrix, int(n-1)), list(range(n)))
    return circ

# Std binary implementation using Bell basis
def get_w_circ(j, n, lamb, T):
    
    circuit = QuantumCircuit(n)
    if j == 0:
        circuit.p(-lamb, j)
        circuit.rx(2 * T, j)
        circuit.p(lamb, j)
        return circuit
    for i in range(j):
        circuit.cx(control_qubit=j, target_qubit=i)
    circuit.p(-lamb, j)
    circuit.h(j)

    # multi_controlled_rz = RZGate(2 * T).control(int(j))
    multi_controlled_rz = get_mcrz(j+1, 2 * T)
    circuit.append(multi_controlled_rz, qargs=np.arange(0, j+1).tolist())

    circuit.h(j)
    circuit.p(lamb, j)
    for i in range(j):
        circuit.cx(control_qubit=j, target_qubit=j-1-i)
    return circuit

def get_v_circ(n, lamb, T, periodic=True, order="forward"):
    N = 2 ** n
    h = 1 / N
    circuit = QuantumCircuit(n)
    
    if order == "forward":
        # First order Trotter
        for j in range(n):
            circuit.append(get_w_circ(j, n, lamb, T / (2 * h)), qargs=np.arange(n).tolist())

        if periodic:
            for j in range(n-1):
                circuit.cx(control_qubit=n-1, target_qubit=j)
            circuit.p(lamb, n-1)
            circuit.h(n-1)
            for j in range(n-1):
                circuit.x(j)

            # multi_controlled_rz = RZGate(2 * T).control(int(n)-1)
            multi_controlled_rz = get_mcrz(n, 2 * T / (2 * h))
            circuit.append(multi_controlled_rz, qargs=np.arange(0, n).tolist())

            for j in range(n-1):
                circuit.x(j)
            circuit.h(n-1)
            circuit.p(-lamb, n-1)
            for j in range(n-1):
                circuit.cx(control_qubit=n-1, target_qubit=n-2-j)
    elif order == "backward":
        if periodic:
            for j in range(n-1):
                circuit.cx(control_qubit=n-1, target_qubit=j)
            circuit.p(lamb, n-1)
            circuit.h(n-1)
            for j in range(n-1):
                circuit.x(j)

            # multi_controlled_rz = RZGate(2 * T).control(int(n)-1)
            multi_controlled_rz = get_mcrz(n, 2 * T / (2 * h))
            circuit.append(multi_controlled_rz, qargs=np.arange(0, n).tolist())

            for j in range(n-1):
                circuit.x(j)
            circuit.h(n-1)
            circuit.p(-lamb, n-1)
            for j in range(n-1):
                circuit.cx(control_qubit=n-1, target_qubit=n-2-j)

        for j in range(n):
            circuit.append(get_w_circ(n-1-j, n, lamb, T / (2 * h)), qargs=np.arange(n).tolist())

    return circuit

In [94]:
n = 6
for j in range(n-1):
    print(j)
print()
for j in range(n-1):
    print(n-2-j)

0
1
2
3
4

4
3
2
1
0


In [95]:
def bell_basis_gate_count_per_trotter_step(n_x, n_p, R, dt):
    N = 2 ** n_x

    '''Contruct circuit'''
    # First n_p qubits are for p
    trot_circuit = QuantumCircuit(n_x + n_p)

    # Hermitian part
    lamb = -np.pi / 2
    # trot_circuit.append(get_v_circ(n_x, lamb, 0.5 * dt, periodic=True, order="forward"), qargs=np.arange(n_p, n_x + n_p).tolist())

    # Anti-Hermitian part (controlled simulation)
    lamb = 0
    # Controlled simulation
    for j in range(n_p - 1):
        second_order_v_circ = QuantumCircuit(n_x)
        # print((np.pi / R) * (2 ** j))
        second_order_v_circ.append(get_v_circ(n_x, lamb, 0.5 * (np.pi / R) * (2 ** j) * dt, order="forward"), qargs=np.arange(n_x).tolist())
        # second_order_v_circ.append(get_v_circ(n_x, lamb, 0.5 * (np.pi / R) * (2 ** j) * dt, order="backward"), qargs=np.arange(n_x).tolist())
        trot_circuit.append(transpile(second_order_v_circ, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[j], np.arange(n_p, n_x+n_p)]).tolist())

    # second_order_v_circ = QuantumCircuit(n_x)
    # # print(- (np.pi / R) * (2 ** (n_p-1)))
    # second_order_v_circ.append(get_v_circ(n_x, lamb, - 0.5 * (np.pi / R) * (2 ** (n_p-1)) * dt, order="forward"), qargs=np.arange(n_x).tolist())
    # second_order_v_circ.append(get_v_circ(n_x, lamb, - 0.5 * (np.pi / R) * (2 ** (n_p-1)) * dt, order="backward"), qargs=np.arange(n_x).tolist())
    # trot_circuit.append(transpile(second_order_v_circ, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[n_p-1], np.arange(n_p, n_x+n_p)]).tolist())

    # # Diagonal part
    # for j in range(n_p-1):
    #     trot_circuit.p(dt * N * (np.pi / R) * (2 ** j), j)
    # trot_circuit.p(-dt * N * (np.pi / R) * (2 ** (n_p - 1)), n_p - 1)

    # # Hermitian part
    # lamb = -np.pi / 2
    # trot_circuit.append(get_v_circ(n_x, lamb, 0.5 * dt, periodic=True, order="backward"), qargs=np.arange(n_p, n_x + n_p).tolist())

    return trot_circuit


    # Compile and optimize circuit
    
    # compiled_circuit = transpile(trot_circuit, basis_gates=['rxx', 'rx', 'ry', 'rz'], optimization_level=3)
    # num_single_qubit_gates, num_two_qubit_gates = 0,0

    # ops = compiled_circuit.count_ops()
    # for op in ops:
    #     if op == "rx" or op == "ry" or op == "rz":
    #         num_single_qubit_gates += ops[op]
    #     elif op == "rxx":
    #         num_two_qubit_gates += ops[op]
    
    # # tket_circuit = qiskit_to_tk(compiled_circuit)
    # # gateset = {OpType.Rx, OpType.Ry, OpType.Rz, OpType.XXPhase}
    # # rebase = auto_rebase_pass(gateset) 
    # # comp = SequencePass([FullPeepholeOptimise(), CommuteThroughMultis(), RemoveRedundancies(), rebase])
    # # comp.apply(tket_circuit)

    # # Gates per Trotter step
    # # num_single_qubit_gates, num_two_qubit_gates = tket_circuit.n_1qb_gates(), tket_circuit.n_2qb_gates()
    # depth = compiled_circuit.depth()
    # return num_single_qubit_gates, num_two_qubit_gates, depth

In [116]:
def bell_basis_optimized(n_x, n_p, R, dt):
    N = 2 ** n_x

    '''Contruct circuit'''
    # First n_p qubits are for p
    trot_circuit = QuantumCircuit(n_x + n_p)

    # Hermitian part
    lamb = -np.pi / 2
    # trot_circuit.append(get_v_circ(n_x, lamb, 0.5 * dt, periodic=True, order="forward"), qargs=np.arange(n_p, n_x + n_p).tolist())

    # Anti-Hermitian part (controlled simulation)
    lamb = 0
    # Controlled simulation
    for i in range(n_p - 1):
        T = 0.5 * (np.pi / R) * (2 ** i) * dt
        n = n_x
        h = 1 / N

        '''get_w_circ'''
        # First order Trotter
        for j in range(n_x):

            if j == 0:
                trot_circuit.p(-lamb, n_p+j)
                trot_circuit.crx(2 * T, i, n_p+j)
                trot_circuit.p(lamb, n_p+j)
            else:
                for k in range(j):
                    trot_circuit.cx(control_qubit=n_p+j, target_qubit=n_p+k)
                trot_circuit.p(-lamb, n_p+j)
                trot_circuit.h(n_p+j)

                # multi_controlled_rz = RZGate(2 * T).control(int(j))
                multi_controlled_rz = get_mcrz(j+1, 2 * T)
                trot_circuit.append(transpile(multi_controlled_rz, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[i], np.arange(n_p, n_p+j+1)]).tolist())

                trot_circuit.h(n_p+j)
                trot_circuit.p(lamb, n_p+j)
                for k in range(j):
                    trot_circuit.cx(control_qubit=n_p+j, target_qubit=n_p+j-1-k)


            # trot_circuit.append(get_w_circ(j, n, lamb, T / (2 * h)), qargs=np.arange(n_p, n_p+n).tolist())

        for j in range(n-1):
            trot_circuit.cx(control_qubit=n_p+n-1, target_qubit=n_p+j)
        trot_circuit.p(lamb, n_p+n-1)
        trot_circuit.h(n_p+n-1)
        for j in range(n-1):
            trot_circuit.x(n_p+j)

        # multi_controlled_rz = RZGate(2 * T).control(int(n)-1)
        multi_controlled_rz = get_mcrz(n, 2 * T / (2 * h))
        trot_circuit.append(transpile(multi_controlled_rz, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[i], np.arange(n_p, n_x+n_p)]).tolist())

        for j in range(n-1):
            trot_circuit.x(n_p+j)
        trot_circuit.h(n_p+n-1)
        trot_circuit.p(-lamb, n_p+n-1)
        for j in range(n-1):
            trot_circuit.cx(control_qubit=n_p+n-1, target_qubit=n_p+n-2-j)


        # for j in range(n-1):
        #     trot_circuit.cx(control_qubit=n_p+n-1, target_qubit=n_p+j)
        # trot_circuit.p(lamb, n_p+n-1)
        # trot_circuit.h(n_p+n-1)
        # for j in range(n-1):
        #     trot_circuit.x(n_p+j)

        # # multi_controlled_rz = RZGate(2 * T).control(int(n)-1)
        # multi_controlled_rz = get_mcrz(n, 2 * T / (2 * h))
        # trot_circuit.append(transpile(multi_controlled_rz, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[i], np.arange(n_p, n_x+n_p)]).tolist())

        # for j in range(n-1):
        #     trot_circuit.x(n_p+j)
        # trot_circuit.h(n_p+n-1)
        # trot_circuit.p(-lamb, n_p+n-1)
        # for j in range(n-1):
        #     trot_circuit.cx(control_qubit=n_p+n-1, target_qubit=n_p+n-2-j)

        # for j in range(n):
        #     trot_circuit.append(get_w_circ(n-1-j, n, lamb, T / (2 * h)), qargs=np.arange(n_p, n_p+n).tolist())


        # second_order_v_circ.append(get_v_circ(n_x, lamb, 0.5 * (np.pi / R) * (2 ** j) * dt, order="forward"), qargs=np.arange(n_x).tolist())
        # second_order_v_circ.append(get_v_circ(n_x, lamb, 0.5 * (np.pi / R) * (2 ** j) * dt, order="backward"), qargs=np.arange(n_x).tolist())
        # trot_circuit.append(transpile(second_order_v_circ, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[j], np.arange(n_p, n_x+n_p)]).tolist())
    
    # T = - 0.5 * (np.pi / R) * (2 ** (n_p-1)) * dt
    # n = n_x
    # h = 1 / N


    # # First order Trotter
    # for j in range(n_x):
    #     trot_circuit.append(get_w_circ(j, n, lamb, T / (2 * h)), qargs=np.arange(n).tolist())

    # for j in range(n-1):
    #     trot_circuit.cx(control_qubit=n-1, target_qubit=j)
    # trot_circuit.p(lamb, n-1)
    # trot_circuit.h(n-1)
    # for j in range(n-1):
    #     trot_circuit.x(j)

    # # multi_controlled_rz = RZGate(2 * T).control(int(n)-1)
    # multi_controlled_rz = get_mcrz(n, 2 * T / (2 * h))
    # trot_circuit.append(transpile(multi_controlled_rz, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[n_p-1], np.arange(n_p, n_x+n_p)]).tolist())

    # for j in range(n-1):
    #     trot_circuit.x(j)
    # trot_circuit.h(n-1)
    # trot_circuit.p(-lamb, n-1)
    # for j in range(n-1):
    #     trot_circuit.cx(control_qubit=n-1, target_qubit=n-2-j)


    # for j in range(n-1):
    #     trot_circuit.cx(control_qubit=n-1, target_qubit=j)
    # trot_circuit.p(lamb, n-1)
    # trot_circuit.h(n-1)
    # for j in range(n-1):
    #     trot_circuit.x(j)

    # # multi_controlled_rz = RZGate(2 * T).control(int(n)-1)
    # multi_controlled_rz = get_mcrz(n, 2 * T / (2 * h))
    # trot_circuit.append(transpile(multi_controlled_rz, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[n_p-1], np.arange(n_p, n_x+n_p)]).tolist())

    # for j in range(n-1):
    #     trot_circuit.x(j)
    # trot_circuit.h(n-1)
    # trot_circuit.p(-lamb, n-1)
    # for j in range(n-1):
    #     trot_circuit.cx(control_qubit=n-1, target_qubit=n-2-j)

    # for j in range(n):
    #     trot_circuit.append(get_w_circ(n-1-j, n, lamb, T / (2 * h)), qargs=np.arange(n).tolist())
    # # second_order_v_circ.append(get_v_circ(n_x, lamb, - 0.5 * (np.pi / R) * (2 ** (n_p-1)) * dt, order="forward"), qargs=np.arange(n_x).tolist())
    # # second_order_v_circ.append(get_v_circ(n_x, lamb, - 0.5 * (np.pi / R) * (2 ** (n_p-1)) * dt, order="backward"), qargs=np.arange(n_x).tolist())
    # # trot_circuit.append(transpile(second_order_v_circ, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=0).control(1), qargs=np.concatenate([[n_p-1], np.arange(n_p, n_x+n_p)]).tolist())

    # # # Diagonal part
    # # for i in range(n_p-1):
    # #     trot_circuit.p(dt * N * (np.pi / R) * (2 ** i), i)
    # # trot_circuit.p(-dt * N * (np.pi / R) * (2 ** (n_p - 1)), n_p - 1)

    # # # Hermitian part
    # # lamb = -np.pi / 2
    # # trot_circuit.append(get_v_circ(n_x, lamb, 0.5 * dt, periodic=True, order="backward"), qargs=np.arange(n_p, n_x + n_p).tolist())

    return trot_circuit

In [138]:
n_x = 2
n_p = 2
R = 10
dt = 0.5


In [139]:
U1 = Operator(bell_basis_gate_count_per_trotter_step(n_x, n_p, R, dt)).to_matrix()

U2 = Operator(bell_basis_optimized(n_x, n_p, R, dt)).to_matrix()

print(np.abs(np.trace(U1 @ np.conj(U2.T))) / (2 ** (n_x + n_p)))

0.9976903760076745


In [140]:
bell_basis_optimized(n_x, n_p, R, dt).draw()

┌──────────────────┐     »
q_0: ─────────────■────────────────────────────────┤0                 ├─────»
                  │                                │                  │     »
q_1: ─────────────┼────────────────────────────────┤                  ├─────»
     ┌──────┐┌────┴─────┐┌──────┐┌───┐             │  c_circuit-31519 │     »
q_2: ┤ P(0) ├┤ Rx(π/20) ├┤ P(0) ├┤ X ├─────────────┤1                 ├─────»
     └──────┘└──────────┘└──────┘└─┬─┘┌──────┐┌───┐│                  │┌───┐»
q_3: ──────────────────────────────■──┤ P(0) ├┤ H ├┤2                 ├┤ H ├»
                                      └──────┘└───┘└──────────────────┘└───┘»
«                                    ┌──────────────────┐                  
«q_0: ───────────────────────────────┤0                 ├──────────────────
«                                    │                  │                  
«q_1: ───────────────────────────────┤                  ├──────────────────
«             ┌───┐┌───┐ ┌───┐       │  c_circuit-31559 │┌───┐        ┌───┐
«q_2: ────────┤ X ├┤ X ├─┤ X ├───────┤1                 ├┤ X ├────────┤ X ├
«     ┌──────┐└─┬─┘└─┬─┘┌┴───┴─┐┌───┐│                  │├───┤┌──────┐└─┬─┘
«q_3: ┤ P(0) ├──■────■──┤ P(0) ├┤ H ├┤2                 ├┤ H ├┤ P(0) ├──■──
«     └──────┘          └──────┘└───┘└──────────────────┘└───┘└──────┘

In [132]:
bell_basis_gate_count_per_trotter_step(n_x, n_p, R, dt).decompose().draw()

q_0: ────────■─────────
             │         
q_1: ────────┼─────────
     ┌───────┴────────┐
q_2: ┤0               ├
     │  circuit-31027 │
q_3: ┤1               ├
     └────────────────┘